In [1]:
import torch

from src.nano_maker.skeleton import SkeletonModel
from src.nano_maker.container import configs

from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

In [2]:
c = configs.skeleton_config
model = SkeletonModel(n_embd=c['n_embd'], n_head=c['n_head'], n_layers=c['n_layers'],
                      block_size=c['block_size'], map4_res=c['map4_res'], radial_resolution=c['radial_resolution'],
                      l_a=c['l_a'], dropout=c['dropout'])

In [20]:
from src.paths import *
sk_wt_path = PROJECT_ROOT / "src/nano_maker/container/prototype_checkpoint_1.6.pt"

In [21]:
prototype_weights = torch.load(sk_wt_path, map_location="cpu")

In [22]:
print(type(prototype_weights))

<class 'dict'>


In [23]:
model.load_state_dict(prototype_weights["model_state_dict"])

<All keys matched successfully>

In [24]:
device = 'cpu'
block_size = c["block_size"]
radial_resolution = c["radial_resolution"]

In [25]:
sample_smiles = "CCNC(=O)[C@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)CCc1ccc(Cl)cc1 |wU:57.63,12.20,wD:45.57,31.44,23.28,5.4,63.67,(30.68,-5.02,;30.68,-6.54,;29.36,-7.34,;28.03,-6.58,;26.73,-7.34,;28,-5.06,;29.2,-4.18,;28.69,-2.75,;27.18,-2.75,;26.75,-4.2,;25.41,-4.94,;25.41,-6.48,;24.09,-4.18,;24.09,-2.64,;25.44,-1.88,;25.44,-.33,;26.79,.42,;26.79,1.95,;28.13,2.73,;25.47,2.74,;22.76,-4.94,;21.41,-4.16,;21.44,-2.62,;20.08,-4.9,;20.06,-6.45,;21.4,-7.24,;21.38,-8.77,;22.73,-6.47,;18.74,-4.13,;17.42,-4.89,;17.41,-6.42,;16.1,-4.12,;16.1,-2.59,;15.64,-1.13,;14.17,-.64,;14.17,.9,;15.65,1.37,;16.26,2.77,;17.81,2.93,;18.71,1.7,;18.07,.28,;16.54,.12,;14.77,-4.87,;13.43,-4.1,;13.43,-2.55,;12.08,-4.86,;12.08,-6.38,;13.4,-7.18,;13.4,-8.72,;14.72,-9.49,;16.07,-8.73,;17.39,-9.51,;16.07,-7.19,;14.75,-6.42,;10.76,-4.07,;9.44,-4.83,;9.41,-6.38,;8.11,-4.06,;8.12,-2.52,;9.44,-1.77,;6.77,-4.81,;5.44,-4.03,;5.45,-2.51,;4.1,-4.8,;4.09,-6.34,;5.42,-7.12,;6.83,-6.5,;7.85,-7.64,;7.06,-8.98,;7.53,-10.44,;6.5,-11.57,;5,-11.24,;4.52,-9.78,;5.57,-8.66,;2.78,-4.03,;1.43,-4.77,;1.45,-6.31,;.08,-4.03,;-1.37,-4.99,;-2.89,-4.2,;-2.98,-2.49,;-4.53,-1.72,;-5.95,-2.65,;-7.46,-1.8,;-5.86,-4.38,;-4.33,-5.15,)|"

sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
sample_skeleton = model.generate(sample_map4)
sample_skeleton

[tensor([[51.1127, 53.7413, 54.2069]], grad_fn=<MulBackward0>),
 tensor([[26.3523, 24.3804, 24.7039]], grad_fn=<MulBackward0>),
 tensor([[19.0350, 24.5340, 32.8959]], grad_fn=<MulBackward0>),
 tensor([[ 3.0166,  6.6178, 10.2145]], grad_fn=<MulBackward0>),
 tensor([[ 4.0065, 22.5564, 35.0320]], grad_fn=<MulBackward0>),
 tensor([[ 5.1616, 33.5280, 40.6121]], grad_fn=<MulBackward0>),
 tensor([[ 6.2970, 41.4150, 42.2689]], grad_fn=<MulBackward0>),
 tensor([[ 7.3332, 44.4648, 43.6810]], grad_fn=<MulBackward0>),
 tensor([[ 8.0761, 44.9837, 43.6873]], grad_fn=<MulBackward0>),
 tensor([[ 8.1649, 44.9598, 42.9338]], grad_fn=<MulBackward0>),
 tensor([[ 7.6650, 45.2575, 42.6195]], grad_fn=<MulBackward0>),
 tensor([[ 7.2972, 45.6492, 43.0633]], grad_fn=<MulBackward0>),
 tensor([[ 7.2334, 45.0056, 42.1158]], grad_fn=<MulBackward0>),
 tensor([[ 6.9830, 44.8711, 41.6395]], grad_fn=<MulBackward0>),
 tensor([[ 6.7560, 44.4044, 41.1648]], grad_fn=<MulBackward0>),
 tensor([[ 6.4718, 45.7602, 40.8407]], g

In [26]:
model.eval()
with torch.no_grad():
    ctx = torch.zeros(1, block_size, 3, device=device)

    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(ctx, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [7.753468036651611, 9.389548301696777, 2.241460084915161]
Caffeine: [3.1862401962280273, 13.572549819946289, 3.41184139251709]
Ibuprofen: [3.942582130432129, 14.684701919555664, 3.8146274089813232]


In [27]:
model.eval()
with torch.no_grad():
    fp = torch.tensor(smiles_fingerprint("CC(=O)Oc1ccccc1C(=O)O"),
                     dtype=torch.float32).unsqueeze(0).to(device)

    # try different context windows - all padding vs partial real context
    ctx_empty = torch.full((1, block_size, 3),
                           float(radial_resolution * 1.5), device=device)
    ctx_mid = torch.full((1, block_size, 3),
                         float(50), device=device)  # mid-sequence
    ctx_late = torch.full((1, block_size, 3),
                          float(10), device=device)  # near end

    for name, ctx in [("empty", ctx_empty), ("mid", ctx_mid), ("late", ctx_late)]:
        out, _ = model(ctx, fp)
        print(f"{name} context: {out.squeeze().tolist()}")

empty context: [50.079193115234375, 49.361610412597656, 51.9390869140625]
mid context: [2.4642770290374756, 2.916513681411743, 2.806009292602539]
late context: [3.435818910598755, 3.2933435440063477, 3.5186991691589355]


In [28]:
# Pocket Skeleton Generation algorithm prototyping

In [29]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker

In [30]:
r_mod = RadialSeeker(100, 100, 33, False)

In [31]:
def process_skeleton(pocket_skeleton):
    output = []
    for vector in pocket_skeleton:
        output.append(vector.detach().flatten().tolist())
    return output

In [32]:
processed_sample = process_skeleton(sample_skeleton)
processed_sample

[[51.112735748291016, 53.74127197265625, 54.206886291503906],
 [26.352252960205078, 24.3803768157959, 24.703886032104492],
 [19.03498649597168, 24.533966064453125, 32.895912170410156],
 [3.0165786743164062, 6.617794990539551, 10.214494705200195],
 [4.006479263305664, 22.556432723999023, 35.031986236572266],
 [5.161622524261475, 33.527950286865234, 40.612083435058594],
 [6.296964645385742, 41.41503143310547, 42.26888656616211],
 [7.3332133293151855, 44.464820861816406, 43.681034088134766],
 [8.076091766357422, 44.98366928100586, 43.687278747558594],
 [8.164924621582031, 44.9598274230957, 42.933841705322266],
 [7.66502046585083, 45.25749969482422, 42.61954116821289],
 [7.297187805175781, 45.64924240112305, 43.06332778930664],
 [7.233440399169922, 45.00562286376953, 42.11577606201172],
 [6.983035564422607, 44.8711051940918, 41.639488220214844],
 [6.75602912902832, 44.40438461303711, 41.16483688354492],
 [6.471807479858398, 45.76023864746094, 40.8406982421875],
 [6.143129348754883, 46.8829

In [33]:
# convert everything into raw angstrom coordinates
angstrom_output = []
for vector in processed_sample:
    angstrom_output.append(r_mod.num2vect(vector))
angstrom_output

[[-0.2655944061279314, 1.4692395019531261, 1.7765449523925767],
 [-16.607513046264646, -17.908951301574707, -17.695435218811035],
 [-21.43690891265869, -17.807582397460937, -12.288697967529295],
 [-32.00905807495117, -29.6322553062439, -27.25843349456787],
 [-31.355723686218262, -19.112754402160643, -10.878889083862305],
 [-30.593329133987428, -11.871552810668945, -7.196024932861327],
 [-29.84400333404541, -6.6660792541503895, -6.102534866333006],
 [-29.16007920265198, -4.653218231201169, -5.170517501831053],
 [-28.669779434204102, -4.31077827453613, -5.166396026611327],
 [-28.61114974975586, -4.326513900756833, -5.663664474487302],
 [-28.941086492538453, -4.130050201416015, -5.871102828979492],
 [-29.183856048583984, -3.871500015258789, -5.578203659057618],
 [-29.22592933654785, -4.296288909912107, -6.203587799072263],
 [-29.39119652748108, -4.385070571899412, -6.517937774658201],
 [-29.541020774841307, -4.693106155395505, -6.83120765686035],
 [-29.728607063293456, -3.7982424926757794

In [34]:
true_sample = []
for vector in angstrom_output:
    appending = True
    for number in vector:
        if abs(number) < 0.33:
            appending = False

    if appending:
        true_sample.append(vector)
    else:
        break
true_sample

[]

In [35]:
print(len(true_sample))

0


In [36]:
model.eval()
with torch.no_grad():
    # exactly what training data looks like at sequence start
    ctx_all_padding = torch.full((1, block_size, 3), 150.0, device=device)

    fp = torch.tensor(smiles_fingerprint("CCO"),
                     dtype=torch.float32).unsqueeze(0).to(device)

    out, _ = model(ctx_all_padding, fp)
    raw = out.detach().flatten().tolist()
    ang = r_mod.num2vect(raw)
    print(f"First prediction (index space): {raw}")
    print(f"First prediction (angstroms):   {ang}")

First prediction (index space): [50.886722564697266, 48.30128860473633, 52.735023498535156]
First prediction (angstroms):   [-0.41476310729980526, -2.1211495208740203, 0.8051155090332074]
